# PPG + ECG → Blood Pressure → Cardiovascular Risk

**Step 2 of the project: training.**

1. Model 1 (`BPNet`, 1D CNN): PPG + ECG → SBP and DBP
2. Model 2 (`RiskNet`, 1D CNN): PPG + ECG + BP from model 1 → No / Low / Moderate / High risk

**How to run on Kaggle**
- Settings → Accelerator: **GPU**, Internet: **On**
- Run all cells
- A checkpoint is saved after every epoch in `/kaggle/working/checkpoints`. If the session stops, run all again and it continues from the last epoch.
  (In a new session, first add the old notebook's output with *Add Input → Notebook Output*.)
- At the end download `bp_cnn.pt` and `risk_cnn.pt` from `/kaggle/working/weights` and push them to the `weights/` folder of the repo.

In [ ]:
import os, sys, glob, shutil

REPO_URL = "https://github.com/NeelakandanNC/final_year_project"

if os.path.exists("models.py"):              # running locally inside the repo
    REPO_DIR = "."
else:                                        # on Kaggle: download the repo (code + data)
    REPO_DIR = "/tmp/final_year_project"
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
sys.path.append(REPO_DIR)
print("Repo folder:", REPO_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from models import BPNet, RiskNet, normalize, RISK_NAMES, BP_MEAN, BP_STD

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print("Device:", device)

## Settings

In [ ]:
EPOCHS_BP = 40
EPOCHS_RISK = 25
BATCH_SIZE = 256
LR = 1e-3

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
CKPT_DIR = f"{OUT_DIR}/checkpoints"
WEIGHTS_DIR = f"{OUT_DIR}/weights"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

## Load the data collected by `collect_data.py`

In [ ]:
files = sorted(glob.glob(f"{REPO_DIR}/data/part_*.npz"))
assert files, "No data found. Run collect_data.py and push the data/ folder first."
parts = [np.load(f) for f in files]
data = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}

X = normalize(data["X"])                       # (N, 2, 1000): PPG, ECG
BP = np.stack([data["sbp"], data["dbp"]], 1)   # (N, 2): SBP, DBP in mmHg
risk = data["risk"]
subject = data["subjectid"]
caseid = data["caseid"]

print("Segments:", len(X), "| Patients:", len(np.unique(caseid)))
print("SBP mean %.1f ± %.1f   DBP mean %.1f ± %.1f" % (BP[:, 0].mean(), BP[:, 0].std(), BP[:, 1].mean(), BP[:, 1].std()))
for r, name in enumerate(RISK_NAMES):
    print(f"{name:14s}: {np.sum(risk == r):6d} segments")

In [ ]:
# one example segment
i = 0
t = np.arange(1000) / 100
fig, ax = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
ax[0].plot(t, data["X"][i, 0]); ax[0].set_ylabel("PPG")
ax[1].plot(t, data["X"][i, 1], color="tab:red"); ax[1].set_ylabel("ECG")
ax[1].set_xlabel("time (s)")
ax[0].set_title(f"SBP {BP[i, 0]:.0f} / DBP {BP[i, 1]:.0f} mmHg   |   {RISK_NAMES[risk[i]]}")
plt.show()

## Train / validation / test split (by patient, so no patient is in two sets)

In [ ]:
subj_risk = pd.Series(risk, index=subject).groupby(level=0).first()
rng = np.random.default_rng(42)
test_subj, val_subj = [], []
for r in range(4):                                  # split each risk class 70 / 15 / 15
    s = subj_risk[subj_risk == r].index.values.copy()
    rng.shuffle(s)
    n = len(s)
    test_subj += list(s[:int(0.15 * n)])
    val_subj += list(s[int(0.15 * n):int(0.30 * n)])

split = np.where(np.isin(subject, test_subj), "test", np.where(np.isin(subject, val_subj), "val", "train"))

def make_loader(name, shuffle):
    m = split == name
    ds = TensorDataset(torch.tensor(X[m]), torch.tensor(BP[m], dtype=torch.float32), torch.tensor(risk[m]))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_dl, val_dl, test_dl = make_loader("train", True), make_loader("val", False), make_loader("test", False)
for name in ["train", "val", "test"]:
    print(f"{name:5s}: {np.sum(split == name):6d} segments")

## Training loop with a checkpoint after every epoch

In [ ]:
def find_checkpoint(name):
    path = f"{CKPT_DIR}/{name}_last.pt"
    if os.path.exists(path):
        return path
    old = glob.glob(f"/kaggle/input/**/{name}_last.pt", recursive=True)   # output of an older run
    if old:
        for f in [old[0], old[0].replace("_last.pt", "_best.pt")]:
            if os.path.exists(f):
                shutil.copy(f, CKPT_DIR)
        return f"{CKPT_DIR}/{name}_last.pt"
    return None


def fit(name, model, batch_loss, epochs):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    start, best, history = 0, float("inf"), {"train": [], "val": []}

    ckpt = find_checkpoint(name)
    if ckpt:
        state = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["optimizer"])
        sched.load_state_dict(state["scheduler"])
        start, best, history = state["epoch"] + 1, state["best"], state["history"]
        print(f"Resuming {name} from epoch {start + 1}")

    for epoch in range(start, epochs):
        model.train()
        train_losses = []
        for batch in train_dl:
            loss = batch_loss(model, batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = np.mean([batch_loss(model, b).item() for b in val_dl])
        train_loss = np.mean(train_losses)
        sched.step(val_loss)
        history["train"].append(train_loss)
        history["val"].append(val_loss)

        msg = ""
        if val_loss < best:
            best = val_loss
            torch.save(model.state_dict(), f"{CKPT_DIR}/{name}_best.pt")
            msg = "  <- best"
        # save to a temp file first so a crash during saving can't break the checkpoint
        torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),
                    "scheduler": sched.state_dict(), "best": best, "history": history},
                   f"{CKPT_DIR}/{name}_tmp.pt")
        os.replace(f"{CKPT_DIR}/{name}_tmp.pt", f"{CKPT_DIR}/{name}_last.pt")
        print(f"{name}  epoch {epoch + 1}/{epochs}  train {train_loss:.4f}  val {val_loss:.4f}{msg}")

    model.load_state_dict(torch.load(f"{CKPT_DIR}/{name}_best.pt", map_location=device))
    plt.figure(figsize=(6, 3))
    plt.plot(history["train"], label="train"); plt.plot(history["val"], label="val")
    plt.title(name); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()
    return model

## Model 1: BP estimation (PPG + ECG → SBP, DBP)

In [ ]:
bp_mean, bp_std = BP_MEAN.to(device), BP_STD.to(device)

def bp_loss(model, batch):
    x, bp, _ = [t.to(device) for t in batch]
    return nn.functional.smooth_l1_loss(model(x), (bp - bp_mean) / bp_std)

bp_model = fit("bp_cnn", BPNet().to(device), bp_loss, EPOCHS_BP)

In [ ]:
bp_model.eval()
pred, true = [], []
with torch.no_grad():
    for x, bp, _ in test_dl:
        pred.append(bp_model.predict_bp(x.to(device)).cpu().numpy())
        true.append(bp.numpy())
pred, true = np.concatenate(pred), np.concatenate(true)

report = ["BP model (test set)"]
for i, name in enumerate(["SBP", "DBP"]):
    err = pred[:, i] - true[:, i]
    within = [np.mean(np.abs(err) <= k) * 100 for k in (5, 10, 15)]
    report.append(f"{name}: MAE {np.abs(err).mean():.2f} mmHg | ME {err.mean():.2f} ± SD {err.std():.2f} mmHg | "
                  f"within 5/10/15 mmHg: {within[0]:.0f}% / {within[1]:.0f}% / {within[2]:.0f}%")
report.append("(AAMI standard: ME within ±5 mmHg and SD below 8 mmHg)")
print("\n".join(report))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, name in enumerate(["SBP", "DBP"]):
    ax[i].scatter(true[:, i], pred[:, i], s=2, alpha=0.3)
    lo, hi = true[:, i].min(), true[:, i].max()
    ax[i].plot([lo, hi], [lo, hi], "r--")
    ax[i].set_xlabel(f"true {name} (mmHg)"); ax[i].set_ylabel(f"predicted {name} (mmHg)"); ax[i].set_title(name)
plt.tight_layout(); plt.show()

## Model 2: Cardiovascular risk (PPG + ECG + predicted BP → 4 classes)

In [ ]:
# freeze the BP model, its predictions are used as an input
bp_model.eval()
for p in bp_model.parameters():
    p.requires_grad = False

# class weights because the risk classes are not balanced
counts = np.bincount(risk[split == "train"], minlength=4)
class_w = torch.tensor(counts.sum() / (4 * np.maximum(counts, 1)), dtype=torch.float32, device=device)
ce = nn.CrossEntropyLoss(weight=class_w)

def risk_loss(model, batch):
    x, _, y = [t.to(device) for t in batch]
    with torch.no_grad():
        bp_pred = bp_model.predict_bp(x)
    return ce(model(x, bp_pred), y)

risk_model = RiskNet().to(device)
risk_model.encoder.load_state_dict(bp_model.encoder.state_dict())   # start from the BP model's features
risk_model = fit("risk_cnn", risk_model, risk_loss, EPOCHS_RISK)

In [ ]:
risk_model.eval()
probs = []
with torch.no_grad():
    for x, _, _ in test_dl:
        x = x.to(device)
        probs.append(torch.softmax(risk_model(x, bp_model.predict_bp(x)), 1).cpu().numpy())
probs = np.concatenate(probs)
y_true = risk[split == "test"]
y_pred = probs.argmax(1)

# patient level: average the predictions of all segments of a patient
df = pd.DataFrame(probs)
df["caseid"] = caseid[split == "test"]
df["true"] = y_true
patient = df.groupby("caseid").mean()
p_true = patient["true"].astype(int).values
p_pred = patient[[0, 1, 2, 3]].values.argmax(1)

report.append("")
report.append("Risk model (test set)")
report.append(f"Segment level: accuracy {accuracy_score(y_true, y_pred) * 100:.1f}% | macro F1 {f1_score(y_true, y_pred, average='macro'):.3f}")
report.append(f"Patient level: accuracy {accuracy_score(p_true, p_pred) * 100:.1f}% | macro F1 {f1_score(p_true, p_pred, average='macro'):.3f}")
print("\n".join(report[-3:]))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (t, p, title) in zip(ax, [(y_true, y_pred, "Segments"), (p_true, p_pred, "Patients")]):
    cm = confusion_matrix(t, p, labels=range(4))
    ConfusionMatrixDisplay(cm, display_labels=RISK_NAMES).plot(ax=a, colorbar=False, xticks_rotation=30)
    a.set_title(title)
plt.tight_layout(); plt.show()

## Save the final weights

In [ ]:
torch.save(bp_model.state_dict(), f"{WEIGHTS_DIR}/bp_cnn.pt")
torch.save(risk_model.state_dict(), f"{WEIGHTS_DIR}/risk_cnn.pt")
with open(f"{WEIGHTS_DIR}/results.txt", "w") as f:
    f.write("\n".join(report) + "\n")

for f in sorted(os.listdir(WEIGHTS_DIR)):
    print(f, f"{os.path.getsize(f'{WEIGHTS_DIR}/{f}') / 1e6:.2f} MB")
print("\nDownload these files and put them in the weights/ folder of the GitHub repo.")